# P7-3 · Video và embodied AI (nâng cao)

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import numpy as np
def sample_frames(n_frames, k, train, rng=np.random.default_rng(0)):
    edges = np.linspace(0, n_frames, k + 1)                     # k đoạn bằng nhau
    if train:
        return [int(rng.uniform(edges[i], edges[i + 1])) for i in range(k)]
    return [int((edges[i] + edges[i + 1]) / 2) for i in range(k)]

sample_frames(150, 16, train=False)   # [4, 14, 23, 32, 42, 51, 60, 70, 79, 89, 98, 107, 117, 126, 135, 145]

[4, 14, 23, 32, 42, 51, 60, 70, 79, 89, 98, 107, 117, 126, 135, 145]


## Bài tập 4: Đặc trưng khung hình + Transformer thời gian {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn

class Anticipator(nn.Module):
    def __init__(self, n_verbs, n_nouns, d_in=512, d=256, T=8, layers=2):
        super().__init__()
        self.proj = nn.Linear(d_in, d)
        self.pos = nn.Parameter(torch.zeros(1, T + 1, d))
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        enc = nn.TransformerEncoderLayer(d, nhead=4, dim_feedforward=4 * d, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc, layers)
        self.verb, self.noun = nn.Linear(d, n_verbs), nn.Linear(d, n_nouns)

    def forward(self, feats):                       # feats: (B, T, d_in)
        x = self.proj(feats)
        x = torch.cat([self.cls.expand(len(x), -1, -1), x], 1) + self.pos
        h = self.encoder(x)[:, 0]
        return self.verb(h), self.noun(h)

m = Anticipator(n_verbs=97, n_nouns=300)
v, n = m(torch.randn(4, 8, 512))
print(v.shape, n.shape)

torch.Size([4, 97]) torch.Size([4, 300])


## Bài tập 5: Lấy mẫu khung hình cho video dài ngắn khác nhau {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np

def deu(n, k=8):
    # TODO
    return np.zeros(k, int)

def theo_doan(n, k, rng):
    # TODO
    return np.zeros(k, int)

In [ ]:
def deu(n, k=8):
    b = np.linspace(0, n, k + 1)
    return ((b[:-1] + b[1:]) / 2).astype(int)

def theo_doan(n, k, rng):
    b = np.linspace(0, n, k + 1).astype(int)
    return rng.integers(b[:-1], b[1:])

rng = np.random.default_rng(0)
for giay in (2, 60):
    n = giay * 30
    print(f"video {giay} s ({n} khung): đều {deu(n).tolist()} | ngẫu nhiên {theo_doan(n, 8, rng).tolist()} | cách nhau {n / 8 / 30:.2f} s")

video 2 s (60 khung): đều [3, 11, 18, 26, 33, 41, 48, 56] | ngẫu nhiên [5, 12, 18, 24, 32, 37, 45, 52] | cách nhau 0.25 s
video 60 s (1800 khung): đều [112, 337, 562, 787, 1012, 1237, 1462, 1687] | ngẫu nhiên [39, 407, 596, 880, 1013, 1261, 1568, 1739] | cách nhau 7.50 s


## Bài tập 6: Dự đoán hành động tiếp theo trong bếp {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
from collections import Counter, defaultdict
train = [["mở", "lấy", "rửa", "cắt", "cho_vào", "khuấy", "đậy", "mở", "khuấy", "lấy"],
         ["lấy", "rửa", "cắt", "cho_vào", "đổ", "khuấy", "nếm", "khuấy", "tắt"],
         ["mở", "lấy", "đặt", "đóng", "lấy", "rửa", "lau", "cắt", "cho_vào", "khuấy"],
         ["lấy", "cắt", "cho_vào", "đổ", "khuấy", "nếm", "thêm", "khuấy", "tắt", "rửa"],
         ["mở", "lấy", "đóng", "rửa", "cắt", "cắt", "cho_vào", "khuấy", "nếm", "tắt"]]
test = [["lấy", "rửa", "cắt", "cho_vào", "khuấy", "nếm", "tắt"], ["mở", "lấy", "đóng", "cắt", "cho_vào", "đổ", "khuấy"]]
# TODO: theo_sau[a] = Counter các hành động ngay sau a; pho_bien = Counter mọi hành động
#       recall_bigram, recall_pho_bien (top-5) trên mọi cặp (hiện tại → tiếp theo) của tập test

In [ ]:
theo_sau, pho_bien = defaultdict(Counter), Counter()
for seq in train:
    pho_bien.update(seq)
    for a, b in zip(seq, seq[1:]):
        theo_sau[a][b] += 1
top_pb = [a for a, _ in pho_bien.most_common(5)]

def top5(hien_tai):
    return [a for a, _ in theo_sau[hien_tai].most_common(5)] or top_pb

cap = [(a, b) for seq in test for a, b in zip(seq, seq[1:])]
recall_bigram = sum(b in top5(a) for a, b in cap) / len(cap)
recall_pho_bien = sum(b in top_pb for a, b in cap) / len(cap)
print(f"{len(cap)} cặp test | bigram {recall_bigram:.2f} | phổ biến nhất {recall_pho_bien:.2f}")
print("sau 'cắt' thường là:", top5("cắt"))

12 cặp test | bigram 0.92 | phổ biến nhất 0.67
sau 'cắt' thường là: ['cho_vào', 'cắt']


## Bài tập 7: Rời rạc hóa hành động robot thành token {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
LO = np.array([-0.05, -0.05, -0.05, -0.3, -0.3, -0.3, 0.0])
HI = np.array([0.05, 0.05, 0.05, 0.3, 0.3, 0.3, 1.0])
BINS = 256
rng = np.random.default_rng(0)
A = rng.uniform(LO, HI, (1000, 7))

def ma_hoa(a):
    # TODO
    return np.zeros(7, int)

def giai_ma(t):
    # TODO
    return np.zeros(7)

In [ ]:
def ma_hoa(a):
    return np.clip(((np.clip(a, LO, HI) - LO) / (HI - LO) * BINS).astype(int), 0, BINS - 1)

def giai_ma(t):
    return LO + (t + 0.5) / BINS * (HI - LO)

sai = np.abs(giai_ma(ma_hoa(A)) - A).max(0)
print("token của hành động đầu:", ma_hoa(A[0]).tolist())
print("sai số lớn nhất từng chiều:", sai.round(5))

token của hành động đầu: [163, 69, 10, 4, 208, 233, 155]
sai số lớn nhất từng chiều: [0.0002  0.0002  0.0002  0.00117 0.00117 0.00117 0.00195]
